# Relation Extraction and Distant Supervision

The narrative companion to formalNLP's twenty-third topic. Every number here comes from
`relation_extraction_distant_supervision.py`, whose harness asserts each claim the page makes. The testbed is the
finance corpus's relation layer: a fixed knowledge base `WORLD` and documents rendered from it, where fact
sentences state relations and *co-mention* sentences mention a pair without stating anything, even when the KB
relates the pair. The classifiers train on 300 documents about Topic 22's 18 training companies and are tested on a
corpus about the same 18 and one about the 6 held-out companies.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("notebooks/relation-extraction-distant-supervision").resolve()))
import relation_extraction_distant_supervision as rx

np.set_printoptions(precision=4, suppress=True)
for name in ("train", "seen", "unseen"):
    print(name, len(rx.corpus(name)), "sentences,", len(rx.instances(name)), "candidates,",
          Counter(rx.instance_kind(name)))

## 1. Relation mentions and distant supervision (Def 1, Def 2, Prop 1)

A candidate is an ordered pair $(i, j)$ of linked mentions in one sentence, and its gold label is the relation the
sentence states from $i$ to $j$, or NA. Distant supervision labels it with $\mathrm{KB}(h, t)$, every relation the
KB holds between the two entities. Each DS label the sentence does not state has exactly one of three sources:
the sentence states nothing that yields it (co-mention), it follows from what the sentence states by the KB's rules
(entailed: BASED_IN from "headquartered in Toronto, Canada"), or the sentence states another relation of the pair
(overlap: ACQUIRED beside PARENT_OF).

In [ ]:
e = rx.ds_error_sources("train")
print({k: e[k] for k in ("labels", "correct", "co-mention", "entailed", "overlap")}, "precision", round(e["precision"], 6))
print("corpus's own count:", round(rx.fc.distant_supervision_precision(rx.docs("train")), 6))
for r, v in e["per_relation"].items():
    print(f"  {r:14s} {v['labels']:4d} labels, precision {v['precision']:.3f}")

## 2. What distant supervision learns (Thm 1, Cor 1)

Under log-loss against the DS label distribution $\ell(r) = 1[r \in L]/|L|$, the risk is minimized pointwise by
$q^*(r \mid \varphi) = \mathbb{E}[\ell(r) \mid \varphi]$. On a template that never states a relation, that is the
template's KB-related rate $\rho$: a property of the corpus, not of the sample size. The fitted DS classifier
reproduces it template by template, and predicts a relation exactly where $\rho\,\pi > 1 - \rho$.

In [ ]:
rx.test_thm1_ds_optimum_is_the_expected_ds_label()
for t in rx.comention_templates():
    decide = "relation" if t["fitted_top"] > t["fitted_na"] else "NA"
    print(f"{t['template'][0]:16s} {t['template'][2]:44s} rho {t['rho']:.3f} pi {t['pi']:.3f} "
          f"fitted {t['fitted']:.3f} -> {decide}")

In [ ]:
for lab in ("gold", "DS"):
    for t in rx.TESTS:
        s = rx.sentence_scores(t, rx.predict(rx.LAB_SPECS[lab], t))
        print(f"{lab:5s} {t:7s} P {s['p']:.3f} R {s['r']:.3f} F1 {s['f1']:.3f}  labeled with a relation:",
              {k: round(v, 3) for k, v in s["kinds"].items()})

The slider of the lab's second tab removes a share $1 - \kappa$ of the related co-mentions from training, which
lowers each template's $\rho$. Each template flips from a relation to NA as $\rho$ crosses its threshold.

In [ ]:
k = rx.kappa_sweep()
print("kappa ", k["kappa"])
print("F1    ", np.round(k["f1"], 3))
print("related labeled", np.round(k["related"], 3))

## 3. At least one witness (Def 3, Thm 2, Cor 2, Prop 2)

A bag is every training mention of one ordered pair. Under the at-least-one assumption the bag's likelihood for a KB
relation $r$ is the noisy-OR $P(\exists) = 1 - \prod_i (1 - p_i(r))$. Fisher's identity makes its gradient an
E-step: each instance is pushed toward $r$ with weight $w_i = p_i / P(\exists)$, and its coefficient
$\prod_{j \ne i}(1 - p_j)/P(\exists)$ vanishes once another instance is a confident witness.

In [ ]:
rx.test_thm2_noisy_or_gradient_is_the_expected_instance_gradient()
rx.test_cor2_one_witness_silences_the_rest()
w = rx.mil_witness_example()
print(w["relation"], w["h"], "->", w["t"], "P(exists)", w["P_any"])
for m in w["mentions"]:
    print(f"  stated={m['stated']!s:5s} MIL {m['mil']:.3f} DS {m['ds']:.3f} w {m['w']:.3f} push {m['push']:.3g}  ",
          " ".join(m["toks"]))

In [ ]:
v = rx.at_least_one_violations()
print("at least one fails for", v["violated"], "of", v["positive"], "positive (bag, relation) pairs:",
      round(v["rate"], 3), "| DS label error", round(v["ds_error"], 3))
for t in rx.TESTS:
    m = rx.sentence_scores(t, rx.predict(rx.LAB_SPECS["MIL"], t))
    print(f"MIL {t:7s} P {m['p']:.3f} R {m['r']:.3f} F1 {m['f1']:.3f}", {k: round(x, 3) for k, x in m["kinds"].items()})

Prop 2 says which templates the bag labels pin down: the always-consistent hypothesis is
$h_{\max} = 1[c \notin N]$, with $N$ the templates of negative bags, and it is the only one iff every template outside
$N$ is the sole such template of some positive bag. MIL's recall is high where the constraints force the answer and
low where they leave it free.

In [ ]:
rx.test_prop2_identifiability_by_brute_force()
for t in rx.TESTS:
    print(t, {k: (round(x, 3) if isinstance(x, float) else x) for k, x in rx.mil_recall_by_identifiability(t).items()})

## 4. From mentions to facts (Def 4, Prop 3)

Facts aggregate a pair's mentions by their maximum or by noisy-OR. Noisy-OR can only exceed the maximum and tends to
1 as weak mentions accumulate, so it overcounts for a classifier trained sentence by sentence; MIL was trained under
it.

In [ ]:
for t in rx.TESTS:
    ins = rx.instances(t)
    for lab in ("gold", "DS", "MIL"):
        P = rx.predict(rx.LAB_SPECS[lab], t)
        row = {agg: {k: round(v, 3) for k, v in rx.fact_scores(rx.facts(ins, P, agg), ins).items() if k in ("p", "r", "f1")}
               for agg in rx.AGGREGATORS}
        print(f"{t:7s} {lab:4s}", row)

## 5. An incomplete KB is positive-unlabeled (Def 5, Thm 3, Prop 4)

If the KB knows each entity pair with probability $c$, independently of the text, the DS optimum is $c$ times the
complete one on every relation. Argmax then keeps a stating template only while $c > 1/2$, and co-mention templates
drop out first, which is why a slightly incomplete KB *raises* DS's sentence F1. Dividing by $c$ restores recall.
Scored against an independent KB of the same rate, predictions show $c$ times their precision.

In [ ]:
for c in rx.KEEP_GRID:
    r = rx.incomplete_kb(c)
    print(f"c {c:.1f}: {r['kb_facts']:3d} facts, mass {r['mass_ratio']:.3f}, e1 {r['e1']:.3f} | raw R {r['raw']['r']:.3f} "
          f"F1 {r['raw']['f1']:.3f} | /c R {r['by_c']['r']:.3f} F1 {r['by_c']['f1']:.3f} | fact P world "
          f"{r['facts_world']['p']:.3f} vs KB {r['facts_kb']['p']:.3f} vs indep {r['facts_indep']['p']:.3f}")

## 6. Finance case study: names held out (Def 6, Prop 5)

Topic 22's headline tagger finds the spans, exact string matching links them to `WORLD`, and the entity-blind
classifiers read the relation. A triple is right only if both arguments were found, which bounds end-to-end recall;
the gold-trained classifier meets the bound. (The first call pretrains Topic 12's encoder, about 21 s.)

In [ ]:
for t in rx.TESTS:
    for sp in rx.SPAN_SOURCES:
        for lab in ("gold", "ds", "mil"):
            r = rx.pipeline(lab, t, sp)
            print(f"{t:7s} {sp:9s} {lab:4s} P {r['p']:.3f} R {r['r']:.3f} F1 {r['f1']:.3f} bound {r['bound']:.3f} | "
                  f"facts P {r['facts']['p']:.3f} R {r['facts']['r']:.3f}")

In [ ]:
for spec in ("DS + argument words", "DS + encoder"):
    for t in rx.TESTS:
        s = rx.sentence_scores(t, rx.predict(rx.LAB_SPECS[spec], t))
        print(f"{spec:20s} {t:7s} related co-mentions labeled {s['kinds']['related']:.3f}")
kb = rx.extracted_kb()
print("extracted KB:", sum(v > 0.5 for v in kb.values()), "facts above 0.5, all true:",
      all(f in rx.WORLD.triples for f, v in kb.items() if v > 0.5))

## Harness

Every assert in the module, then the summary the page quotes.

In [ ]:
rx._run_harness()